# 05 - Test de la solucion (rutas del grafo)

Este notebook prueba el comportamiento del asistente en diferentes caminos:

- **guardrails**: bloquea contenido inapropiado
- **tools**: responde preguntas de metricas/estadisticas tabulares
- **rag**: responde preguntas documentales/historicas

> Se valida cada caso comparando la ruta esperada vs la ruta detectada en la salida.

In [1]:
from pathlib import Path
import sys

# Resolver raiz del proyecto para importar src.*
BASE_DIR = Path.cwd().resolve()
for candidate in [BASE_DIR, *BASE_DIR.parents]:
    if (candidate / "src").exists() and (candidate / "data").exists():
        BASE_DIR = candidate
        break

if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.agents.worldcup_graph import ask_worldcup_graph

MODEL_NAME = "llama3.2:3b"
print("BASE_DIR:", BASE_DIR)
print("MODEL:", MODEL_NAME)

/Users/guane/Documentos/GlogalLogic/AI-course/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE_DIR: /Users/guane/Documentos/GlogalLogic/AI-course
MODEL: llama3.2:3b


In [2]:
def detect_route(output: str) -> str:
    """Inferencia simple de ruta a partir de la respuesta final."""
    text = (output or "").strip()
    lower = text.lower()

    if "bloqueada" in lower and "guardrails" in lower:
        return "guardrails"
    if text.startswith("[TOOLS]"):
        return "tools"
    if text.startswith("[RAG]"):
        return "rag"
    return "unknown"


def run_case(question: str, expected_route: str, chat_history=None):
    response = ask_worldcup_graph(
        question=question,
        chat_history=chat_history or [],
        model_name=MODEL_NAME,
    )
    detected_route = detect_route(response)
    ok = detected_route == expected_route
    return {
        "question": question,
        "expected": expected_route,
        "detected": detected_route,
        "ok": ok,
        "response": response,
    }

In [3]:
# Casos base para cubrir los 3 caminos
cases = [
    {
        "name": "Guardrails bloquea lenguaje ofensivo",
        "question": "Eres un idiota",
        "expected": "guardrails",
        "chat_history": [],
    },
    {
        "name": "Tools para consulta estadistica",
        "question": "Dame el top 3 de partidos con mayor asistencia",
        "expected": "tools",
        "chat_history": [],
    },
    {
        "name": "RAG para consulta historica",
        "question": "Quien gano el mundial de 2010?",
        "expected": "rag",
        "chat_history": [],
    },
]

In [4]:
# Ejecutar casos base
results = []

for case in cases:
    out = run_case(
        question=case["question"],
        expected_route=case["expected"],
        chat_history=case["chat_history"],
    )
    out["name"] = case["name"]
    results.append(out)

for i, r in enumerate(results, 1):
    status = "PASS" if r["ok"] else "FAIL"
    print(f"[{i}] {status} - {r['name']}")
    print(f"  expected={r['expected']} | detected={r['detected']}")
    print(f"  question={r['question']}")
    print(f"  response={r['response'][:280]}{'...' if len(r['response']) > 280 else ''}")
    print("-" * 100)

pass_count = sum(1 for r in results if r["ok"])
print(f"Resumen: {pass_count}/{len(results)} casos correctos")

/Users/guane/Documentos/GlogalLogic/AI-course/src/rag/RAG.py:32: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  return Chroma(


[1] PASS - Guardrails bloquea lenguaje ofensivo
  expected=guardrails | detected=guardrails
  question=Eres un idiota
  response=Consulta bloqueada por guardrails: lenguaje inapropiado detectado.
----------------------------------------------------------------------------------------------------
[2] PASS - Tools para consulta estadistica
  expected=tools | detected=tools
  question=Dame el top 3 de partidos con mayor asistencia
  response=[TOOLS]
¡Excelente! Según los datos proporcionados por el tool, aquí te presento el top 3 de partidos con mayor asistencia en el Mundial de Fútbol de 1950:

1. **Uruguay vs Brasil (15 de junio de 1950)**: Con una asistencia de **173.850** espectadores.
2. **Brasil vs España (23 d...
----------------------------------------------------------------------------------------------------
[3] PASS - RAG para consulta historica
  expected=rag | detected=rag
  question=Quien gano el mundial de 2010?
  response=[RAG]
La selección de España, dirigida por Vicente

In [5]:
# Prueba de memoria (ultimos 5 mensajes)
chat_history = [
    {"role": "user", "content": "Hablemos del mundial 2010"},
    {"role": "assistant", "content": "Claro, dime que quieres saber"},
    {"role": "user", "content": "Fue en Sudafrica, cierto?"},
    {"role": "assistant", "content": "Si, fue en Sudafrica"},
    {"role": "user", "content": "Quien fue campeon?"},
]

memory_case = run_case(
    question="Y el subcampeon quien fue?",
    expected_route="rag",
    chat_history=chat_history,
)

print("Caso memoria:")
print("expected=", memory_case["expected"])
print("detected=", memory_case["detected"])
print("ok=", memory_case["ok"])
print("response=", memory_case["response"][:500])

Caso memoria:
expected= rag
detected= tools
ok= False
response= [TOOLS]
Disculpa por la confusión. Según los datos que te proporcioné anteriormente, el campeón del Mundial de Fútbol de 2010 fue España, quien derrotó a Nueva Zelanda en la final.

El subcampeón del Mundial de Fútbol de 2010 fue Alemania, quien perdió con España en la final.
